In [13]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib
import openpyxl

### Reading the data in and creating dataframes

In [2]:
# read all data in the data folder; will take ~4 minutes to run 
path = "data/"

xl = pd.ExcelFile(os.path.join(path, "2015-2025 SPD Calls with Close Codes.xlsx")) 

# for the SPD Outcomes data, which includes an unnessary "KEY" sheet that we will ignore
sheets = [sheet for sheet in xl.sheet_names if sheet != "KEY"]

print("Reading CSV")
print('2015...')
eug_cad2015 = pd.read_csv(os.path.join(path, "EugeneCAD2015noloc.csv"))
print('2016...')
eug_cad2016 = pd.read_csv(os.path.join(path, "EugeneCAD2016noloc.csv"))
print('2017...')
eug_cad2017 = pd.read_csv(os.path.join(path, "EugeneCAD2017noloc.csv"))
print('2018...')
eug_cad2018 = pd.read_csv(os.path.join(path, "EugeneCAD2018noloc.csv"))
print('2019...')
eug_cad2019 = pd.read_csv(os.path.join(path, "EugeneCAD2019noloc.csv"))
print('2020...')
eug_cad2020 = pd.read_csv(os.path.join(path, "EugeneCAD2020noloc.csv"))
print('2021...')
eug_cad2021 = pd.read_csv(os.path.join(path, "EugeneCAD2021noloc.csv"), low_memory=False)
print('2022...')
eug_cad2022 = pd.read_csv(os.path.join(path, "EugeneCAD2022noloc.csv"))
print('2023...')
eug_cad2023 = pd.read_csv(os.path.join(path, "EugeneCAD2023noloc.csv"))
print('2024...')
eug_cad2024 = pd.read_csv(os.path.join(path, "EugeneCAD2024noloc.csv"))
print('2025...')
eug_cad2025 = pd.read_csv(os.path.join(path, "EugeneCAD2025noloc.csv"), low_memory=False)

print("Reading Excel")
print('SPD Calls for Service...')
spd_calls = pd.concat(pd.read_excel(os.path.join(path, '2015-2025 SPD Calls for Service.xlsx'), sheet_name=None), ignore_index=True)
print('SPD Responding Units...')
spd_units = pd.concat(pd.read_excel(os.path.join(path, '2015-2025 SPD Responding Units.xlsx'), sheet_name=None), ignore_index=True)
print("SPD Outcomes...")
spd_outcomes = pd.concat(pd.read_excel(os.path.join(path, '2015-2025 SPD Calls with Close Codes.xlsx'), sheet_name=sheets), ignore_index=True)
print("Done")

Reading CSV
2015...
2016...
2017...
2018...
2019...
2020...
2021...
2022...
2023...
2024...
2025...
Reading Excel
SPD Calls for Service...
SPD Responding Units...
SPD Outcomes...
Done


In [3]:
dfs = [
    eug_cad2015, eug_cad2016, eug_cad2017, eug_cad2018,
    eug_cad2019, eug_cad2020, eug_cad2021, eug_cad2022,
    eug_cad2023, eug_cad2024, eug_cad2025
]

base_cols = set(dfs[0].columns)

for i, df in enumerate(dfs):
    cols = set(df.columns)
    missing = base_cols - cols
    extra = cols - base_cols
    
    if missing or extra:
        print(f"DataFrame {2015 + i}:")
        if missing:
            print("  Missing:", missing)
        if extra:
            print("  Extra:", extra)

# drop that extra column
eug_cad2025 = eug_cad2025.drop(columns={"month"})

DataFrame 2025:
  Extra: {'month'}


In [4]:
eug_cad = eug_cad2015
for df in dfs[1:]:
    eug_cad = pd.concat([eug_cad, df], ignore_index=True)

eug_cad.columns
eug_cad.head()
print(eug_cad.shape)

(1446014, 20)


### Cleaning Eugene CAD Dataset

#### 1. Subset to the columns I need

In [5]:
columns = ["calltime", "nature", "closed_as", "primeunit"]
eug = eug_cad[columns].copy()

eug

,calltime,nature,closed_as,primeunit
0,2015-01-01 00:00:00.000,PERSON STOP,ASSISTED,_5E48
1,2015-01-01 00:00:44.000,FIGHT,RESOLVED,_3F65
2,2015-01-01 00:01:05.000,CHECK WELFARE,ASSISTED,_3J79
3,2015-01-01 00:03:16.000,SHOTS FIRED,PATROL CHECK,_5E48
4,2015-01-01 00:03:34.000,ILLEGAL FIREWORKS,ADVISED,_5K97
...,...,...,...,...
1446009,2025-12-31 23:39:12.000,TRAFFIC STOP,SOBRIETY CHECK,_4U41
1446010,2025-12-31 23:40:48.000,TRAFFIC STOP,UNIFORM TRAFFIC CITATION ISSUED,_5E56
1446011,2025-12-31 23:45:57.000,PATROL CHECK,UNIFORM TRAFFIC CITATION ISSUED,_4E48
1446012,2025-12-31 23:49:12.000,ASSIST OREGON STATE POLICE,ASSISTED,_5T81


#### 2. Turn ```calltime``` into a datetime object

In [6]:
eug["calltime"] = pd.to_datetime(eug["calltime"])
eug.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1446014 entries, 0 to 1446013
Data columns (total 4 columns):
 #   Column     Non-Null Count    Dtype         
---  ------     --------------    -----         
 0   calltime   1446014 non-null  datetime64[ns]
 1   nature     1445961 non-null  object        
 2   closed_as  1422086 non-null  object        
 3   primeunit  1091731 non-null  object        
dtypes: datetime64[ns](1), object(3)
memory usage: 44.1+ MB


#### 3. Subset dataset further to only have welfare check calls

In [7]:
eug = eug[(eug["nature"] == "CHECK WELFARE") | (eug["nature"] == "CHECK WELFARE, CAHOOTS")]

eug

,calltime,nature,closed_as,primeunit
2,2015-01-01 00:01:05,CHECK WELFARE,ASSISTED,_3J79
27,2015-01-01 00:38:37,CHECK WELFARE,ASSISTED,_3J79
47,2015-01-01 01:28:40,CHECK WELFARE,UNABLE TO LOCATE,_5B44
55,2015-01-01 01:41:40,CHECK WELFARE,ASSISTED,_3J79
186,2015-01-01 15:09:21,CHECK WELFARE,ASSISTED,_3J79
...,...,...,...,...
1445916,2025-12-31 17:12:02,CHECK WELFARE,DISREGARDED BY DISPATCH,NaN
1445927,2025-12-31 17:56:18,CHECK WELFARE,UNABLE TO LOCATE,_4U72
1445946,2025-12-31 20:23:13,CHECK WELFARE,WELFARE CHECK DONE,_5E56
1445957,2025-12-31 20:50:01,CHECK WELFARE,DISREGARD,_4U72


### Cleaning Springfield Call Dataset

#### 1. Join SPD Calls and Outcome sets

In [17]:
spd_outcomes[30:].head()

,Incident Number,Close Code
30,15150678,ABAN
31,15201625,ABAN
32,15276199,ABAN
33,15008945,ACC
34,15030803,ACC


In [ ]:
spd = spd_calls.merge(spd_outcomes, on="Incident Number")
spd

#### 2. Subset to columns I need and rename

In [ ]:
columns = ["Final Call Type", "Call Creation Time", "Primary Responding Unit", "Close Code"]
spd = spd[columns].copy()

spd.rename(columns={
    "Final Call Type": "nature",
    "Call Creation Time": "calltime",
    "Primary Responding Unit": "primeunit",
    "Close Code": "closed_as"
}, inplace=True)

spd = spd[spd["nature"] == "CHECK WELFARE"]

#### 3. Turn ```calltime``` into a datetime object

In [ ]:
spd["calltime"] = pd.to_datetime(spd["calltime"])
spd.info()

#### 4. Clean the ```primeunit``` and ```closed_as``` column

In [ ]:
spd["primeunit"] = spd["primeunit"].str.strip()
spd["closed_as"] = spd["closed_as"].str.strip()

In [ ]:
spd

### CSV Output

In [ ]:
eug.to_csv("cleaned_eug.csv", index=False)
spd.to_csv("cleaned_spd.csv", index=False)